In [2]:
import torch

ckpt_path = "/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_berk/2025-12-05T191217_First film try same with patch size 256/checkpoints/model_best.pth" 
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

print("Top-level keys:", ckpt.keys())


Top-level keys: dict_keys(['arch', 'epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'best_metric', 'best_epoch', 'config', 'wandb_id', 'logdir', 'run_name', 'scaler_state_dict'])


In [3]:
print(ckpt["best_metric"])
print(ckpt["best_epoch"])

print("\n=== CONFIG ===")
for k,v in ckpt["config"].items():
    print(k, ":", v)


0.688046353861404
77

=== CONFIG ===
random_seed : 19
gpu : 0
data.dataset : PanNuke
data.dataset_path : /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke
data.input_shape : 256
data.train_folds : [0]
data.val_folds : [1]
data.num_nuclei_classes : 6
data.num_tissue_classes : 1
dataloader.train.num_workers : 6
dataloader.train.pin_memory : True
dataloader.train.persistent_workers : True
dataloader.train.drop_last : True
dataloader.val.num_workers : 4
dataloader.val.pin_memory : True
dataloader.val.persistent_workers : True
dataloader.val.drop_last : True
model.backbone : sam-h-rosie-film
model.pretrained_encoder : /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/CellViT-plus-plus/checkpoints/SAM/encoder_only_CellViT-SAM-H-x40-AMP.pth
model.rosie_hidden_dim : 256
fusion.freeze_cellvit : True
fusion.freez

In [4]:
print("==== Searching for metric names in checkpoint ====")
for k in ckpt.keys():
    if "metric" in k.lower() or "score" in k.lower():
        print(k, ":", ckpt[k])


==== Searching for metric names in checkpoint ====
best_metric : 0.688046353861404


In [5]:
print("\n==== Searching config for metric names ====")
for k,v in ckpt["config"].items():
    if "metric" in k.lower() or "monitor" in k.lower():
        print(k, ":", v)



==== Searching config for metric names ====


Connect to wandb to get the metrics

In [6]:
import wandb

run = wandb.Api().run("samh-rosie-film/k68ttwdk")

# Print available summary metrics
print("=== SUMMARY METRICS ===")
for k,v in run.summary.items():
    print(k, ":", v)

# Print history keys
print("\n=== HISTORY KEYS ===")
history = run.history(samples=5)
print(history.columns)


=== SUMMARY METRICS ===
Best-Epoch : 77
Best-Metric : 0.688046353861404
Binary-Cell-Dice-Mean/Train : 0.9167978673708213
Binary-Cell-Dice-Mean/Validation : 0.8097800939704813
Binary-Cell-Jacard-Mean/Train : 0.8507988192556825
Binary-Cell-Jacard-Mean/Validation : 0.6973950930617072
Example-Predictions/Train : {'_type': 'image-file', 'format': 'png', 'height': 600, 'path': 'media/images/Example-Predictions/Train_80_5fffc55ef50e9ce88a8e.png', 'sha256': '5fffc55ef50e9ce88a8e86d5911921760d684193b3b21d3d9dda4ec9d5898443', 'size': 255716, 'width': 900}
Example-Predictions/Validation : {'_type': 'image-file', 'format': 'png', 'height': 1200, 'path': 'media/images/Example-Predictions/Validation_80_da284a3d9f407275d90e.png', 'sha256': 'da284a3d9f407275d90eb00bec925ed9f0082f437f194f1a278085d4b9ac745f', 'size': 1160824, 'width': 900}
Learning-Rate/Learning-Rate : 1.0007709637592772e-05
Loss/Train : 3.1750598628165196
Loss/Validation : 3.4648467302322388
Tissue-Multiclass-Accuracy/Train : 1
Tissue-

In [7]:
hist = run.history()

epoch_77 = hist[hist["_step"] == 77]
epoch_77


,hv_map_msge/Train,lung-bPQ/Validation,Loss/Train,lung-mPQ/Validation,Example-Predictions/Validation,Learning-Rate/Learning-Rate,nuclei_binary_map_dice/Validation,nuclei_binary_map_dice/Train,macrophage-PQ/Validation,nuclei_binary_map_bce/Validation,...,Binary-Cell-Jacard-Mean/Validation,Example-Predictions/Train,hv_map_mse/Validation,mPQ/Validation,lung-Jaccard/Validation,neutrophil-PQ/Validation,tissue_types_ce/Train,lung-Dice/Validation,nuclei_type_map_dice/Train,tissue_types_ce/Validation
76,0.161226,0.683074,3.188467,0.601959,"{'size': 762529, 'height': 1200, 'width': 900,...",0.00001,0.215248,0.117796,0.424398,0.252089,...,0.708908,"{'size': 313614, 'height': 600, 'width': 900, ...",0.024353,0.601959,0.708908,0.420952,0,0.819427,1.500885,0


# Imports & Setup

In [8]:
import torch
import wandb
import pandas as pd
from pathlib import Path


## Helper to inspect checkpoint

In [15]:
def load_checkpoint_info(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = ckpt["config"]

    return {
        "path": ckpt_path,
        "file": Path(ckpt_path).name,
        "run_id": cfg["logging.run_id"],
        "project": cfg["logging.project"],
        "best_epoch": ckpt["best_epoch"],
        "best_metric": ckpt["best_metric"],
        "config": cfg
    }


## Get W&B metrics for a given run and epoch

In [10]:
def extract_metrics_from_wandb(run_id, project, best_epoch):
    api = wandb.Api()
    run = api.run(f"{project}/{run_id}")

    hist = run.history()  # full history
    row = hist[hist["_step"] == best_epoch]

    if len(row) == 0:
        print(f"No row found for epoch {best_epoch}")
        return None

    # Take the first (and only) matching row
    row = row.iloc[0].to_dict()
    return row


# Metrics we want in the comparison table

In [11]:
METRICS_OF_INTEREST = [
    "Binary-Cell-Dice-Mean/Validation",
    "Binary-Cell-Jacard-Mean/Validation",
    "mPQ/Validation",
    "epithelial-PQ/Validation",
    "lymphocyte-PQ/Validation",
    "macrophage-PQ/Validation",
    "neutrophil-PQ/Validation",
    "bPQ/Validation",
    "hv_map_mse/Validation",
    "hv_map_msge/Validation",
]


In [16]:
def compare_checkpoints(checkpoint_paths):
    rows = []

    for ckpt_path in checkpoint_paths:
        info = load_checkpoint_info(ckpt_path)

        metrics = extract_metrics_from_wandb(
            run_id=info["run_id"],
            project=info["project"],
            best_epoch=info["best_epoch"]
        )

        row_data = {
            "checkpoint": info["file"],
            "best_epoch": info["best_epoch"],
            "best_metric(IoU)": info["best_metric"],
        }

        for m in METRICS_OF_INTEREST:
            row_data[m] = metrics.get(m, None)

        rows.append(row_data)

    return pd.DataFrame(rows)


In [17]:
checkpoint_paths = [
    "/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_berk/2025-12-05T191217_First film try same with patch size 256/checkpoints/model_best.pth",
    "/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p128_pannuke/logs_local/2025-12-04T002122_First film try/checkpoints/model_best.pth",
    "/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p128_pannuke/logs_local/2025-12-03T180258_tcga_finetune_128/checkpoints/model_best.pth",
    #"/path/to/cellvit128.ckpt",
]

df = compare_checkpoints(checkpoint_paths)
df


,checkpoint,best_epoch,best_metric(IoU),Binary-Cell-Dice-Mean/Validation,Binary-Cell-Jacard-Mean/Validation,mPQ/Validation,epithelial-PQ/Validation,lymphocyte-PQ/Validation,macrophage-PQ/Validation,neutrophil-PQ/Validation,bPQ/Validation,hv_map_mse/Validation,hv_map_msge/Validation
0,model_best.pth,77,0.688046,0.819427,0.708908,0.601959,0.655383,0.535017,0.424398,0.420952,0.683074,0.024353,0.207391
1,model_best.pth,30,0.618261,0.787342,0.678431,0.555259,0.579502,0.535735,0.377375,0.415182,0.612954,0.026868,0.222584
2,model_best.pth,25,0.620713,0.786073,0.675100,0.562372,0.587800,0.520749,0.380716,0.474678,0.615624,0.028500,0.219525
